# 🩺 第二十六天 · CBLUE 第一个机器学习模型（KUAKE-QIC）

**今天目标（约 1.5 小时）**：用 **TF-IDF + 逻辑回归**跑通 KUAKE-QIC（医疗查询意图分类，11 类），在验证集上拿到你的第一个「机器学习准确率」。

**预期结果（助手已提前验证）**：dev 准确率 **75.7%**——比「全猜最多类」的 34.6% 高出一倍多。

> ⚠️ sklearn 已装好（base 环境，清华镜像）。先 **Kernel → Restart Kernel**。

## ⚠️ 先记住这堂课：中文分词的坑（今天最重要的知识）

如果你直接用默认的 `TfidfVectorizer()`，得到的是 **36.2%**（几乎等于瞎猜）；改成 `analyzer='char'` 才到 **75.7%**。原因：

- 默认分词器按**英文空格/标点**切词，而中文句子没有空格，整句被当成一个"词"，特征全废；
- 中文要靠**字符 n-gram**（`analyzer='char'`）或专业分词库（jieba）。

**这一步是所有中文 NLP 的地基**——你以后做任何中文医疗文本任务都会用到。

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

train = pd.read_json("KUAKE-QIC/KUAKE-QIC_train.json")
dev   = pd.read_json("KUAKE-QIC/KUAKE-QIC_dev.json")
test  = pd.read_json("KUAKE-QIC/KUAKE-QIC_test.json")
print(train.shape, dev.shape, test.shape)

(6931, 3) (1955, 3) (1994, 3)


## 第 1 步 · TF-IDF 向量化 + 逻辑回归（跑这个 cell）

- `analyzer='char'`：按**字**切（中文关键一步）；`ngram_range=(1,2)`：同时看单字和双字组合（"血糖""血压"这种词就靠双字捕获）；
- 逻辑回归 = 多分类的"最小可用机器学习模型"，`max_iter` 调大避免收敛警告。

In [2]:
vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 2), min_df=2)
X_train = vec.fit_transform(train["query"])   # 训练集：fit + transform
X_dev   = vec.transform(dev["query"])         # 验证集：只 transform（不能 fit！）

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, train["label"])

pred = clf.predict(X_dev)
print("dev 准确率: %.4f" % accuracy_score(dev["label"], pred))
print(classification_report(dev["label"], pred, digits=3, zero_division=0))

dev 准确率: 0.7570
              precision    recall  f1-score   support

          其他      0.525     0.658     0.584       395
        功效作用      0.933     0.500     0.651        28
        医疗费用      1.000     0.820     0.901        50
        后果表述      0.531     0.378     0.442        45
        就医建议      0.895     0.828     0.860       134
        指标解读      1.000     0.156     0.270        32
        治疗方案      0.845     0.914     0.878       676
        注意事项      0.786     0.675     0.726       120
        疾病表述      0.736     0.582     0.650       158
        病因分析      1.000     0.483     0.651        29
        病情诊断      0.841     0.788     0.814       288

    accuracy                          0.757      1955
   macro avg      0.827     0.617     0.675      1955
weighted avg      0.774     0.757     0.754      1955



## 第 2 步 · 读分类报告，做错误分析（写进周报）

看 `classification_report` 的 **f1-score** 一列：
- **强类**（>0.8）：医疗费用、治疗方案、就医建议、病情诊断——特征词明显（"多少钱""怎么办""是什么病"）；
- **弱类**（<0.5）：**指标解读（0.27）、后果表述（0.44）**——训练样本最少（137/235 条）且语义模糊，模型几乎找不出来。

**你的观察**（写在这里）：
1. 哪个类最弱？为什么？  其他是最弱的，因为描述最模糊，特征最少
2. 如果想提升，你第一反应会改什么？（提示：样本量 / 特征 / 模型）我会使用排除法，用特征先判断他是不是其他以外的种类，如果不是那它就是其他

In [3]:
# 第 3 步 · 预测测试集，生成提交文件
test = test.copy()
test["label"] = clf.predict(vec.transform(test["query"]))
submission = test[["id", "query", "label"]]
submission.to_json("KUAKE-QIC/KUAKE-QIC_test_pred.json", orient="records", force_ascii=False)
print("提交文件已生成，共", len(submission), "条")
print(submission.head(3))

提交文件已生成，共 1994 条
   id            query label
0  s1  黑苦荞茶的功效与作用及食用方法  功效作用
1  s2          交界痣会凸起吗    其他
2  s3      检查是否能怀孕挂什么科  就医建议


## ✅ D26 完成标准（打钩）

- [ ] dev 准确率跑到 **75% 以上**
- [ ] 分类报告看懂：能说出最强的 2 个类和最弱的 2 个类
- [ ] 错误分析写了你的观察
- [ ] 提交文件 `KUAKE-QIC_test_pred.json` 生成成功
- [ ] 保存（**Cmd + S**）

> 完成后喊我验收。**提交说明**：KUAKE-QIC 是 CBLUE 经典任务、榜单已静态，我们以「本地跑通 + 75.7% 基线」作为简历/GitHub 记录；想要活跃榜单就等年底 CCKS/CHIP 2026 报名。

**D27 预告**：简历终稿 + 润色 GitHub README（把「CBLUE 75.7% 基线」这条补进简历和仓库首页）。

---

# 📝 D26 学习笔记（大白话通关版）· 面试前 5 分钟翻这页

## 一句话总流程
> 句子 → 按字切分 → 剔除稀有特征 → 翻译成 TF-IDF 数字 → 乘上模型学好的权重打分 → 取最高分 → 翻回文字 = 答案

## 九个必会知识点
1. **中文必须 `analyzer='char'`**：默认按英文空格分词，中文没空格会被当成一整串 → 特征全废（36% vs 75.7%）。
2. **`ngram_range=(1,2)`**：单字和双字组合都算特征（"血糖""血压"靠双字捕获）。
3. **`min_df=2` 防过拟合**：df = 出现某特征的**句子数**（不是次数）；只在 1 句出现的就是噪声，砍掉（例：32682→11589 个特征），省内存还防模型记住没用的怪特征。
4. **TF-IDF = 词频 × 稀有度**：TF = 这句里出现得多不多；IDF = 全局稀不稀有（"怎么"烂大街→权重低，"降压"稀有→权重高）。合起来筛出"既多又稀有"的有价值词。
5. **fit / transform / fit_transform**：fit = 编字典（看数据决定哪些字算特征 + 算 IDF）；transform = 拿字典把句子翻成数字；fit_transform = 两步合一。**字典只编一次（用训练集）**。
6. **测试集绝不能 fit（数据泄漏）**：fit = 从数据建立规则，测试集必须假装"还没来的未来数据"，只能借训练集编好的字典（transform）。fit 了测试集 = 偷看答案 → 分数虚高、真实能力虚。
7. **precision / recall / f1（医学版）**：recall = 灵敏度（真有病的抓出几成，漏诊）；precision = 阳性预测值（报阳的几成真病，误报）；f1 = 两者调和平均。分类报告按类看：recall 低 = 它自己被漏；precision 低 = 它被别人冒名。
8. **两套权重别混**：TF-IDF 权重 = vec 算好的**静态**权重（有公式，存 X 里）；coef_ 权重 = clf **迭代学出来**的动态权重（从随机起步、每轮往错误小的方向调一点点，收敛即完成；max_iter=2000 是耐心上限）。
9. **评估三件套**：`pred` = 模型答卷（**文字**，predict 内部打分→选编号→查 `classes_` 翻回文字）；`dev["label"]` = 标准答案；`accuracy_score(标准答案, pred)` = 对答案数比例——**必须两个参数**，光有 pred 自己比自己是 1.0，毫无意义。

## 生命周期速记（sklearn 万物同构）
```
TfidfVectorizer: 创建(带配置) → fit(装字典) → transform(翻译成数字)
LogisticRegression: 创建(带配置) → fit(编号classes_ + 学权重coef_) → predict(打分→classes_翻回文字)
```

## 面试高频追问（都能答）
- TF-IDF 是什么？→ 词频 × 稀有度
- 为什么测试集不能 fit？→ 数据泄漏 = 偷看答案
- pred 是文字还是数字？→ 文字，predict 内部编号经 classes_ 翻译后输出
- classes_ 什么时候生成？→ fit 时；predict 拿它把编号翻回文字
- coef_ 为什么是 11×11589？→ 11 个类各一套权重 × 11589 个特征
- precision/recall 用医学话说？→ 阳性预测值 / 灵敏度

## 我的错误分析要点（写周报用）
- 强类（f1>0.8）：医疗费用、治疗方案、就医建议、病情诊断
- 弱类：指标解读（0.27，recall 0.156 → 32 句只抓 5 句 = 漏诊）、后果表述（0.44）
- 规律：弱类 = 训练样本最少 + 语义模糊 → 与 D23 评测"类别不均衡影响表现"同构

## 简历一句话（D27 用）
「在 CBLUE KUAKE-QIC（医疗查询意图分类，11 类）上用 TF-IDF + 逻辑回归实现基线，dev 准确率 75.7%（多数类基线 34.6%）；掌握中文 NLP 特征工程与分类评估方法论。」